In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.shape

(898, 35)

In [3]:
df.head()  # 34 features, 1 category

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [4]:
df["Class"].nunique()

7

In [5]:
X = df.drop(["Class"], axis=1)
y = df["Class"]

In [6]:
from sklearn.preprocessing import LabelEncoder , StandardScaler

le = LabelEncoder()
y = le.fit_transform(y)

In [7]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X , y , test_size = 0.2 , random_state=42 
)

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN (Classifier)

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset

In [10]:
X_train_tensor = torch.tensor(X_train_scaled , dtype=torch.float32)
y_train_tensor = torch.tensor(y_train , dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled , dtype=torch.float32)
y_test_tensor = torch.tensor(y_test , dtype=torch.long)

In [11]:
train_dataset = TensorDataset(X_train_tensor , y_train_tensor)
test_dataset = TensorDataset(X_test_tensor , y_test_tensor)

In [12]:
train_loader = DataLoader(train_dataset , batch_size=32, shuffle= True)
test_loader = DataLoader(test_dataset , batch_size=32 , shuffle = True)

In [13]:
# Model Architechture

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1],64),      # 1st hidden layer
            nn.ReLU(),
            nn.Linear(64,64),              # 2nd hidden layer
            nn.ReLU(),
            nn.Linear(64,7)                # 3rd hidden layer
        )

    def forward(self,x):
        return self.model(x)

In [14]:
model = ANN()
# Loss fnx
criterion = nn.CrossEntropyLoss()     # Calculates both Loss and SOFTMAX AF
optimizer = optim.Adam(model.parameters())

In [15]:
# Training Model
epochs = 100 

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 

    for xbatch , ybatch in train_loader:
        optimizer.zero_grad()

        output = model(xbatch)
        loss = criterion(output , ybatch)
        loss.backward()
        optimizer.step()     # Params update

        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    
    print(f"epoch {epoch+1}/{epochs} Training loss: {train_loss}")

epoch 1/100 Training loss: 1.7109061738719111
epoch 2/100 Training loss: 1.1118444385735884
epoch 3/100 Training loss: 0.7880214996959852
epoch 4/100 Training loss: 0.5865093391874562
epoch 5/100 Training loss: 0.45966956796853436
epoch 6/100 Training loss: 0.3854260075351466
epoch 7/100 Training loss: 0.34348705929258594
epoch 8/100 Training loss: 0.30883893953717273
epoch 9/100 Training loss: 0.2842511729053829
epoch 10/100 Training loss: 0.26170552230399585
epoch 11/100 Training loss: 0.24763531464597452
epoch 12/100 Training loss: 0.2293878071334051
epoch 13/100 Training loss: 0.22113107825103012
epoch 14/100 Training loss: 0.21408602854479913
epoch 15/100 Training loss: 0.2032622111880261
epoch 16/100 Training loss: 0.18406068274508353
epoch 17/100 Training loss: 0.17632961143618045
epoch 18/100 Training loss: 0.1773237501797469
epoch 19/100 Training loss: 0.1719248395251191
epoch 20/100 Training loss: 0.16891554864528385
epoch 21/100 Training loss: 0.1613516153200813
epoch 22/100

In [16]:
# Evaluation 
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb , yb in test_loader:
        output = model(xb)      # [0.2, 0.5, 1.3, -0.5, ..] - 7 vals    (Linear AF will give)
        _,predicted = torch.max(output , 1)       
        
        correct += (predicted == yb).sum().item()
        total += yb.size(0)    # Actual samples in each batch

print("Accuracy : " , correct/total)

Accuracy :  0.9666666666666667


### PCA

In [17]:
X_scaled = scaler.fit_transform(X)
X.shape

(898, 34)

In [18]:
from sklearn.decomposition import PCA

pca = PCA(n_components=24)
X_pca = pca.fit_transform(X_scaled)

print("Explain variance ratio: " , pca.explained_variance_ratio_)

Explain variance ratio:  [4.10025048e-01 2.29233266e-01 1.13888989e-01 6.21067961e-02
 4.98935300e-02 3.78306002e-02 2.67881542e-02 1.80225756e-02
 1.43732889e-02 8.02924749e-03 6.47177311e-03 4.85055132e-03
 4.23093208e-03 3.51545882e-03 2.77988259e-03 2.01951547e-03
 1.40544567e-03 1.40153551e-03 1.16339928e-03 7.35466360e-04
 5.22492480e-04 3.70412789e-04 1.53869028e-04 1.13246640e-04]


In [19]:
X_pca.shape

(898, 24)

In [20]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(
    X_pca , y , test_size = 0.2 , random_state=42 
)

In [21]:
X_train_tensor = torch.tensor(X_train , dtype=torch.float32)
y_train_tensor = torch.tensor(y_train , dtype=torch.long)

X_test_tensor = torch.tensor(X_test , dtype=torch.float32)
y_test_tensor = torch.tensor(y_test , dtype=torch.long)

In [22]:
train_dataset = TensorDataset(X_train_tensor , y_train_tensor)
test_dataset = TensorDataset(X_test_tensor , y_test_tensor)

In [23]:
train_loader = DataLoader(train_dataset , batch_size=32, shuffle= True)
test_loader = DataLoader(test_dataset , batch_size=32 , shuffle = True)

In [24]:
# Model Architechture

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_pca.shape[1],64),      # 1st hidden layer
            nn.ReLU(),
            nn.Linear(64,64),              # 2nd hidden layer
            nn.ReLU(),
            nn.Linear(64,7)                # 3rd hidden layer
        )

    def forward(self,x):
        return self.model(x)

In [25]:
model = ANN()
# Loss fnx
criterion = nn.CrossEntropyLoss()     # Calculates both Loss and SOFTMAX AF
optimizer = optim.Adam(model.parameters())

In [26]:
# Training Model
epochs = 100 

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 

    for xbatch , ybatch in train_loader:
        optimizer.zero_grad()

        output = model(xbatch)
        loss = criterion(output , ybatch)
        loss.backward()
        optimizer.step()     # Params update

        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    
    print(f"epoch {epoch+1}/{epochs} Training loss: {train_loss}")

epoch 1/100 Training loss: 1.7623183468113774
epoch 2/100 Training loss: 1.2281897068023682
epoch 3/100 Training loss: 0.8332828918228978
epoch 4/100 Training loss: 0.6149504482746124
epoch 5/100 Training loss: 0.4886238199213277
epoch 6/100 Training loss: 0.4064519586770431
epoch 7/100 Training loss: 0.3509001090474751
epoch 8/100 Training loss: 0.29992108241371485
epoch 9/100 Training loss: 0.27038952513881354
epoch 10/100 Training loss: 0.23650363346804743
epoch 11/100 Training loss: 0.21671678611765738
epoch 12/100 Training loss: 0.19989584353954895
epoch 13/100 Training loss: 0.18556320148965585
epoch 14/100 Training loss: 0.16907265150676604
epoch 15/100 Training loss: 0.1669368421577889
epoch 16/100 Training loss: 0.15985489393705907
epoch 17/100 Training loss: 0.14341314282754195
epoch 18/100 Training loss: 0.13746312114855516
epoch 19/100 Training loss: 0.13205903622767198
epoch 20/100 Training loss: 0.12635442420192386
epoch 21/100 Training loss: 0.12075372428997704
epoch 22/

In [27]:
# Evaluation 
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb , yb in test_loader:
        output = model(xb)
        _,predicted = torch.max(output , 1)
        
        correct += (predicted == yb).sum().item()
        total += yb.size(0)    # Samples in each batch

print("Accuracy : " , correct/total)

Accuracy :  0.95
